# Lead AI · MLOps

**Objectif :** Tests, CI/CD avec GitHub Actions, entraînement et surveillance continus.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Tests unitaires avec pytest

Un modèle en production, c'est du code : préparation des données, variables, API, règles métier. Sans tests, chaque modification est un pari. Un Lead AI fait tester le code de son équipe, et sait aussi tester ce qui est propre au ML.

## pytest en cinq minutes

Un test est une fonction dont le nom commence par `test_`, qui contient des `assert`. On lance `pytest` dans le terminal : il trouve tous les fichiers `test_*.py` et exécute les tests.

```python
# tests/test_prix.py
import pytest
from monprojet.prix import prix_ttc

def test_cas_simple():
    assert prix_ttc(100) == 120.0

@pytest.mark.parametrize("ht, remise, attendu", [(100, 0.5, 60.0), (0, 0, 0.0), (9.99, 0, 11.99)])
def test_plusieurs_cas(ht, remise, attendu):
    assert prix_ttc(ht, remise=remise) == attendu

def test_ht_negatif():
    with pytest.raises(ValueError):
        prix_ttc(-1)

@pytest.fixture
def modele_entraine():           # préparé une fois, réutilisé par les tests qui le demandent
    return entrainer(charger_petit_jeu())

def test_modele_mieux_que_hasard(modele_entraine):
    assert modele_entraine.score(X_test, y_test) > 0.6
```

## Que tester ?

- **Les cas normaux**, les **cas limites** (zéro, liste vide, valeur maximale) et les **erreurs attendues**.
- Un bon test échoue quand le code est faux. Pour mesurer la qualité d'une suite de tests, on peut introduire volontairement des bugs (des **mutants**) et vérifier que les tests les détectent : c'est le **test de mutation** (outil `mutmut`). Tu vas le faire à la main.

## Tester du machine learning

Le résultat d'un modèle n'est pas exact au centime près. On teste donc des **propriétés** :

- **Le code de données** : `preparer` donne les bonnes colonnes, sans valeur manquante, sans fuite de la cible.
- **Le comportement du modèle** : un appartement plus grand, toutes choses égales par ailleurs, doit coûter plus cher (test **directionnel**) ; changer un détail sans importance ne doit pas changer la prédiction (test d'**invariance**) ; le modèle doit battre une référence simple.
- **Le contrat de l'API** : codes HTTP, validation, format des réponses (avec `TestClient`, vague 6).

La **couverture** (`pytest --cov`) indique quelles lignes sont exécutées par les tests : utile pour repérer les zones oubliées, mais 100 % de couverture ne garantit pas de bons tests.

> **À retenir** : des fonctions `test_*` avec des `assert`, `parametrize` pour les variantes, `raises` pour les erreurs, des fixtures pour la préparation. Un bon test tue les mutants. Pour le ML : tests de données, tests directionnels et d'invariance, comparaison à une référence.

### Exercice 1 · Des tests qui tuent les mutants

La fonction `prix_ttc` et l'utilitaire `doit_lever` (l'équivalent de `pytest.raises`) sont fournis. Écris au moins 4 fonctions de test `test_...` qui passent avec cette fonction. La vérification va ensuite remplacer `prix_ttc` par 5 versions boguées (arrondi au dixième au lieu du centième, absence d'erreur sur un prix négatif, remise ignorée, remise supérieure à 1 acceptée, TVA par défaut à 21 %) : chacune doit faire échouer au moins un de tes tests.

<details><summary>Indice</summary>

Pense à un prix qui distingue un arrondi au dixième (9.99 → 11.988), à une remise de 0,5, et utilise doit_lever pour les deux erreurs (prix négatif, remise 1,5).
</details>

In [ ]:
def prix_ttc(ht, tva=0.2, remise=0):
    """Prix TTC arrondi au centime. ht >= 0 et 0 <= remise <= 1, sinon ValueError."""
    if ht < 0:
        raise ValueError("Le prix hors taxe doit être positif.")
    if not 0 <= remise <= 1:
        raise ValueError("La remise doit être entre 0 et 1.")
    return round(ht * (1 - remise) * (1 + tva), 2)

def doit_lever(exception, fonction, *args, **kwargs):
    """Équivalent de « with pytest.raises(exception): fonction(*args, **kwargs) »."""
    try:
        fonction(*args, **kwargs)
    except exception:
        return
    raise AssertionError(f"{fonction.__name__}{args} aurait dû lever {exception.__name__}")

def test_cas_simple():
    assert prix_ttc(100) == 120.0

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
_original = prix_ttc
_tests = {n: f for n, f in list(globals().items()) if n.startswith("test_") and callable(f)}
assert len(_tests) >= 4, f"Écris au moins 4 fonctions test_… ; tu en as {len(_tests)}."
for _n, _t in _tests.items():
    try:
        _t()
    except Exception as _e:
        raise AssertionError(f"{_n} échoue avec la fonction correcte : {_e}")
def _m1(ht, tva=0.2, remise=0):
    if ht < 0: raise ValueError
    if not 0 <= remise <= 1: raise ValueError
    return round(ht * (1 - remise) * (1 + tva), 1)
def _m2(ht, tva=0.2, remise=0):
    if not 0 <= remise <= 1: raise ValueError
    return round(ht * (1 - remise) * (1 + tva), 2)
def _m3(ht, tva=0.2, remise=0):
    if ht < 0: raise ValueError
    if not 0 <= remise <= 1: raise ValueError
    return round(ht * (1 + tva), 2)
def _m4(ht, tva=0.2, remise=0):
    if ht < 0: raise ValueError
    return round(ht * (1 - remise) * (1 + tva), 2)
def _m5(ht, tva=0.21, remise=0):
    if ht < 0: raise ValueError
    if not 0 <= remise <= 1: raise ValueError
    return round(ht * (1 - remise) * (1 + tva), 2)
_survivants = []
for _nom, _m in [("arrondi au dixième", _m1), ("prix négatif accepté", _m2), ("remise ignorée", _m3), ("remise > 1 acceptée", _m4), ("TVA par défaut à 21 %", _m5)]:
    globals()["prix_ttc"] = _m
    _tue = False
    for _t in _tests.values():
        try:
            _t()
        except Exception:
            _tue = True; break
    if not _tue: _survivants.append(_nom)
globals()["prix_ttc"] = _original
assert not _survivants, f"Ces mutants passent tous tes tests : {', '.join(_survivants)}. Ajoute un test qui les détecte."
print("Bravo, tout est correct.")

### Exercice 2 · Tester le comportement d'un modèle

Écris `verifier_comportement(predire)` : `predire` prend un dictionnaire `{"surface", "pieces", "quartier"}` et renvoie un prix. Renvoie la liste (dans cet ordre) des propriétés **non respectées** parmi : `"directionnel"` (passer de 40 à 80 m², toutes choses égales, augmente le prix, pour chaque quartier de `QUARTIERS`) ; `"positif"` (le prix est strictement positif pour toutes les combinaisons de `SURFACES`, `PIECES` et `QUARTIERS`) ; `"deterministe"` (deux appels identiques donnent le même résultat) ; `"invariance_casse"` (le quartier écrit en minuscules donne le même prix).

<details><summary>Indice</summary>

Une vérification par propriété, chacune avec all(...) ou any(...). itertools.product(SURFACES, PIECES, QUARTIERS) donne toutes les combinaisons.
</details>

In [ ]:
import itertools

QUARTIERS = ["Centre", "Gare", "Port"]
SURFACES = [15, 40, 120]
PIECES = [1, 3, 6]

def verifier_comportement(predire):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import random
PRIX = {"centre": 9800, "gare": 7600, "port": 6400}
bon = lambda a: a["surface"] * PRIX[a["quartier"].lower()] + 5000 * a["pieces"]
assert verifier_comportement(bon) == [], "Le bon modèle respecte toutes les propriétés."
inverse = lambda a: 2_000_000 - a["surface"] * PRIX[a["quartier"].lower()]
assert verifier_comportement(inverse) == ["directionnel"], f"Un modèle où le prix baisse avec la surface viole « directionnel » ; tu renvoies {verifier_comportement(inverse)}."
negatif = lambda a: a["surface"] * PRIX[a["quartier"].lower()] - 200000
assert verifier_comportement(negatif) == ["positif"], "Un modèle qui renvoie des prix négatifs pour les petites surfaces viole « positif »."
_rng = random.Random(0)
bruite = lambda a: bon(a) + _rng.random()
assert "deterministe" in verifier_comportement(bruite), "Un modèle aléatoire viole « deterministe »."
casse = lambda a: a["surface"] * {"Centre": 9800, "Gare": 7600, "Port": 6400}.get(a["quartier"], 5000) + 1
assert verifier_comportement(casse) == ["invariance_casse"], "Un modèle sensible aux majuscules viole « invariance_casse »."
print("Bravo, tout est correct.")

---
## CI/CD avec GitHub Actions

L'**intégration continue** (CI) lance automatiquement les tests à chaque modification proposée : une pull request qui casse quelque chose est repérée avant d'être fusionnée. Le **déploiement continu** (CD) publie automatiquement ce qui est fusionné. **GitHub Actions** fait les deux, directement dans GitHub. Cette plateforme est d'ailleurs publiée sur GitHub Pages par un workflow GitHub Actions.

## Un workflow

Un fichier YAML dans `.github/workflows/` décrit **quand** lancer (événements), **où** (une machine virtuelle) et **quoi** (des étapes).

```python
# .github/workflows/ci.yml
name: CI
on:
  push:
    branches: [main]
  pull_request:

jobs:
  tests:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]          # les tests tournent sur les deux versions
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -r requirements.txt
      - run: pytest --cov

  deploiement:
    needs: tests                                   # seulement si les tests passent
    if: github.ref == 'refs/heads/main'           # seulement sur main, pas sur les PR
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: docker build -t $REGISTRE/fraude-api:${{ github.sha }} .
      - run: docker push $REGISTRE/fraude-api:${{ github.sha }}
        env:
          REGISTRE: ${{ secrets.REGISTRE }}       # les secrets sont stockés dans GitHub, jamais dans le code
```

## Un pipeline MLOps typique

1. **À chaque pull request** : lint (`ruff`), tests unitaires, tests du contrat de l'API, éventuellement un mini-entraînement sur un petit échantillon pour vérifier que rien ne casse.
2. **À la fusion sur main** : construction de l'image Docker, étiquetée avec le commit, poussée dans le registre, déploiement en recette.
3. **Sur validation** (manuelle, ou automatique si les métriques sont bonnes) : déploiement en production.
4. **Sur planification** (`on: schedule: - cron: "0 3 * * 1"`) : réentraînement hebdomadaire, évaluation, rapport.

Des outils comme **CML** (de l'équipe DVC) publient même les métriques et graphiques d'un entraînement en commentaire de la pull request.

> **À retenir** : `on` (quand), `jobs` (quoi), `runs-on` (où), `steps` (comment). `needs` et `if` pour enchaîner tests puis déploiement sur main. Les secrets dans GitHub. Un pipeline ML ajoute l'entraînement et l'évaluation automatiques.

**À faire :**

- [ ] Ajouter à un de tes projets un workflow qui lance pytest à chaque push et pull request
- [ ] Ouvrir une pull request qui casse un test et constater qu'elle est marquée en échec
- [ ] Ajouter un job de construction d'image Docker, seulement sur main
- [ ] Lire le workflow .github/workflows/pages.yml de ce dépôt Prépa AI et expliquer chaque étape

### Exercice 1 · Écrire un workflow

Écris le dictionnaire Python `workflow`, l'équivalent du YAML d'un workflow GitHub Actions : nom `CI` ; déclenché par un `push` sur la branche `main` et par toute `pull_request` (valeur `None`) ; un job `tests` sur `ubuntu-latest`, avec une matrice `python-version` à `["3.11", "3.12"]` et quatre étapes dans l'ordre : `actions/checkout@v4`, `actions/setup-python@v5` avec `python-version` égal à `${{ matrix.python-version }}`, `pip install -r requirements.txt` et `pytest` ; un job `deploiement` qui dépend de `tests` (`needs`) et ne tourne que si `github.ref == 'refs/heads/main'`.

<details><summary>Indice</summary>

Recopie la structure du YAML de la leçon en dictionnaires et listes Python. Les clés avec un tiret (runs-on) sont de simples chaînes.
</details>

In [ ]:
workflow = {
    "name": "CI",
    "on": {},
    "jobs": {},
}

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
w = workflow
assert w["name"] == "CI", "Le workflow s'appelle CI."
assert w["on"].get("push") == {"branches": ["main"]} and "pull_request" in w["on"], "Déclencheurs : push sur main, et pull_request."
t = w["jobs"].get("tests")
assert t and t["runs-on"] == "ubuntu-latest", "Le job tests tourne sur ubuntu-latest."
assert t["strategy"]["matrix"]["python-version"] == ["3.11", "3.12"], "La matrice teste Python 3.11 et 3.12."
s = t["steps"]
assert s[0].get("uses") == "actions/checkout@v4", "La première étape récupère le code (actions/checkout@v4)."
assert s[1].get("uses") == "actions/setup-python@v5" and s[1].get("with", {}).get("python-version") == "${{ matrix.python-version }}", "La deuxième installe Python avec la version de la matrice."
assert [x.get("run") for x in s[2:4]] == ["pip install -r requirements.txt", "pytest"], "Puis on installe les dépendances et on lance pytest."
d = w["jobs"].get("deploiement")
assert d and d.get("needs") in ("tests", ["tests"]), "Le déploiement dépend du job tests (needs)."
assert d.get("if") == "github.ref == 'refs/heads/main'", "Le déploiement ne tourne que sur main."
print("Bravo, tout est correct.")

---
## Entraînement continu et mise en production

Un modèle se dégrade avec le temps : les clients changent, les fraudeurs s'adaptent, les prix bougent. L'**entraînement continu** (*continuous training*) réentraîne le modèle régulièrement ou quand un signal l'exige, et le met en production seulement s'il est meilleur, sans risque pour les utilisateurs.

## Quand réentraîner ?

- **Sur planification** : chaque semaine ou chaque mois. Simple, mais parfois trop tôt ou trop tard.
- **Sur déclencheur** : quand la dérive des données dépasse un seuil (leçon suivante), quand la performance mesurée baisse, ou quand assez de nouvelles données étiquetées sont arrivées.

## Champion contre challenger

Le modèle en production est le **champion**. Le nouveau est le **challenger**. On les compare sur les **mêmes données récentes**, avec des **portes de qualité** (*quality gates*) explicites :

- le challenger gagne au moins X points sur la métrique principale ;
- il ne dépasse pas le budget de latence ou de coût ;
- il ne dégrade pas l'équité entre groupes (par exemple, le rappel sur les clients étrangers et sur les clients français) ;
- il passe les tests de comportement.

Tout cela s'automatise dans le pipeline (MLflow pour les métriques, l'alias `champion` du registre pour la promotion).

## Déployer sans risque

| Stratégie | Principe |
| --- | --- |
| **Shadow** (ombre) | Le challenger reçoit le trafic réel en parallèle, mais ses réponses ne sont pas utilisées : on compare sans risque |
| **Canary** | Une petite part du trafic (1 %, puis 5 %, 25 %…) part vers la nouvelle version ; on surveille, puis on élargit ou on revient en arrière |
| **Blue/green** | Deux environnements complets ; on bascule tout d'un coup, et on peut rebasculer instantanément |
| **Test A/B** | Deux versions sur deux groupes, assez longtemps pour mesurer un effet métier (taux de conversion) avec un test statistique |

*Animation interactive « canary » : à voir sur la plateforme web.*

Pour un canary ou un test A/B, chaque utilisateur doit toujours voir la **même** version : on le répartit selon un **hachage** de son identifiant, pas au hasard à chaque requête.

> **À retenir** : réentraîner sur planification ou sur déclencheur. Comparer champion et challenger sur les mêmes données avec des portes de qualité écrites à l'avance. Déployer en ombre ou en canary, avec retour arrière immédiat.

### Exercice 1 · Les portes de qualité

Écris `decider(champion, challenger, gain_min=0.01, latence_max=200, equite_max=0.05)` : chaque modèle est un dictionnaire avec `auc`, `latence_p95_ms` et `ecart_equite`. Renvoie le tuple `(promouvoir, raisons)` : `raisons` liste, dans cet ordre, `"gain AUC insuffisant"` si l'AUC du challenger est inférieure à celle du champion + `gain_min`, `"latence trop élevée"` si sa latence dépasse `latence_max`, `"écart d'équité trop grand"` si son écart dépasse `equite_max` ; `promouvoir` vaut True seulement s'il n'y a aucune raison.

<details><summary>Indice</summary>

Trois if indépendants qui ajoutent une raison ; renvoie (not raisons, raisons).
</details>

In [ ]:
def decider(champion, challenger, gain_min=0.01, latence_max=200, equite_max=0.05):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ch = {"auc": 0.90, "latence_p95_ms": 50, "ecart_equite": 0.02}
assert decider(ch, {"auc": 0.92, "latence_p95_ms": 80, "ecart_equite": 0.03}) == (True, []), "Un challenger meilleur partout doit être promu."
assert decider(ch, {"auc": 0.905, "latence_p95_ms": 80, "ecart_equite": 0.03}) == (False, ["gain AUC insuffisant"]), "Un gain de 0,005 est inférieur au gain minimal de 0,01."
assert decider(ch, {"auc": 0.95, "latence_p95_ms": 250, "ecart_equite": 0.08}) == (False, ["latence trop élevée", "écart d'équité trop grand"]), "Les raisons doivent être listées dans l'ordre : AUC, latence, équité."
assert decider(ch, {"auc": 0.95, "latence_p95_ms": 250, "ecart_equite": 0.0}, latence_max=300)[0] is True, "Les seuils sont des paramètres."
print("Bravo, tout est correct.")

### Exercice 2 · Un canary stable

Écris `version_pour(id_utilisateur, part_canary)` : calcule un « seau » entre 0 et 99 avec `int(hashlib.md5(id_utilisateur.encode()).hexdigest(), 16) % 100`, et renvoie `"canary"` si le seau est strictement inférieur à `part_canary × 100`, sinon `"stable"`. Puis calcule `part_observee`, la part de canary obtenue pour les 10 000 utilisateurs `"u0"` à `"u9999"` avec `part_canary = 0.05`.

<details><summary>Indice</summary>

hashlib.md5(texte.encode()).hexdigest() donne un hexadécimal ; int(…, 16) le convertit en entier.
</details>

In [ ]:
import hashlib

def version_pour(id_utilisateur, part_canary):
    pass

part_observee = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import hashlib
assert version_pour("alice", 0.3) == version_pour("alice", 0.3), "Un même utilisateur doit toujours avoir la même version."
ids = [f"u{i}" for i in range(10000)]
assert part_observee is not None and abs(part_observee - 0.05) < 0.01, f"Avec 5 %, environ 500 utilisateurs sur 10 000 doivent être en canary ; tu as {part_observee}."
assert all(version_pour(u, 0.2) == "canary" for u in ids[:2000] if version_pour(u, 0.05) == "canary"), "En élargissant de 5 % à 20 %, un utilisateur déjà en canary doit y rester (c'est l'avantage des seaux)."
assert all(version_pour(u, 0) == "stable" for u in ids[:500]) and all(version_pour(u, 1) == "canary" for u in ids[:500]), "Avec 0 %, personne en canary ; avec 100 %, tout le monde."
s = int(hashlib.md5(b"bob").hexdigest(), 16) % 100
assert version_pour("bob", (s + 1) / 100) == "canary" and version_pour("bob", s / 100) == "stable", "Le seau doit être comparé strictement à part_canary × 100."
print("Bravo, tout est correct.")

---
## Surveillance continue et dérive

Un modèle validé à 0,96 d'AUC peut tomber à 0,85 six mois plus tard, sans qu'aucune ligne de code n'ait changé. C'est le monde qui a changé. La **surveillance continue** détecte ces changements avant qu'ils ne coûtent cher.

## Deux types de dérive

- **Dérive des données** (*data drift*) : la distribution des entrées change. Plus de paiements depuis l'étranger, des montants plus élevés. Elle se mesure **sans étiquettes**, tout de suite.
- **Dérive de concept** (*concept drift*) : la relation entre les entrées et la cible change. Les fraudeurs adoptent un nouveau mode opératoire sur une catégorie jusque-là sûre. Les entrées peuvent sembler normales : il faut les **vraies étiquettes**, qui arrivent souvent avec retard (une fraude est confirmée des semaines plus tard).

On surveille donc les deux : les distributions des variables et des prédictions, chaque jour ; la performance réelle, dès que les étiquettes arrivent.

## Mesurer la dérive

*Animation interactive « drift » : à voir sur la plateforme web.*

- Le **PSI** (*population stability index*) découpe la variable de référence en 10 intervalles de même effectif et compare les proportions :

$$
\mathrm{PSI} = \sum_{i} (a_i - r_i) \ln\frac{a_i}{r_i}
$$

Règle d'usage : moins de 0,1, stable ; de 0,1 à 0,2, à surveiller ; au-delà de 0,2, dérive importante.

- Le **test de Kolmogorov-Smirnov** compare deux distributions numériques et donne une p-valeur. Attention : sur de grands volumes, il détecte des écarts minuscules sans importance pratique. On le combine avec une mesure d'ampleur comme le PSI.
- Pour une variable catégorielle : le test du khi-deux, ou le PSI sur les modalités.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
from paiements import charger

avant, apres = charger(mois=0), charger(n=4000, mois=5)
for col in ["montant", "heure", "nb_tx_24h"]:
    p = ks_2samp(avant[col], apres[col]).pvalue
    print(f"{col:10s} p-valeur KS = {p:.2e}")
print(f"paiements depuis l'étranger : {(avant.pays_carte != avant.pays_ip).mean():.1%} → {(apres.pays_carte != apres.pays_ip).mean():.1%}")

## Les outils

**Evidently** (open source, celui du programme) génère des rapports de dérive et de qualité en quelques lignes et s'intègre dans un tableau de bord ou un pipeline. Alternatives : **NannyML** (qui estime même la performance sans étiquettes), **WhyLabs**, **Arize**, ou les moniteurs de **SageMaker**.

```python
from evidently import Report
from evidently.presets import DataDriftPreset
rapport = Report([DataDriftPreset()])
resultat = rapport.run(reference_data=avant, current_data=apres)
resultat.save_html("derive.html")
```

> **À retenir** : dérive des données (sans étiquettes, immédiate) et dérive de concept (avec étiquettes, en retard). PSI pour l'ampleur, KS pour la significativité. Surveiller entrées, prédictions et performance, et relier les alertes au réentraînement.

### Exercice 1 · Le PSI

Écris `psi(reference, actuel, n_bins=10)` : calcule les bords des intervalles avec `np.quantile(reference, np.linspace(0, 1, n_bins + 1))`, remplace le premier bord par −∞ et le dernier par +∞, compte les proportions de chaque échantillon dans ces intervalles (`np.histogram`), remplace les proportions inférieures à 1e-4 par 1e-4 (`np.clip`), puis renvoie Σ (a − r) · ln(a / r) sous forme de `float`.

<details><summary>Indice</summary>

np.histogram(x, bords)[0] donne les effectifs ; divise par len(x) pour les proportions.
</details>

In [ ]:
import numpy as np

def psi(reference, actuel, n_bins=10):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
ref, act = rng.normal(0, 1, 3000), rng.normal(0.4, 1.2, 2000)
b = np.quantile(ref, np.linspace(0, 1, 11)); b[0], b[-1] = -np.inf, np.inf
r = np.clip(np.histogram(ref, b)[0] / 3000, 1e-4, None); a = np.clip(np.histogram(act, b)[0] / 2000, 1e-4, None)
attendu = float(np.sum((a - r) * np.log(a / r)))
v = psi(ref, act)
assert v is not None and abs(v - attendu) < 1e-9, f"PSI attendu {attendu:.4f}, obtenu {v}."
assert psi(ref, ref) < 1e-9, "Le PSI d'une distribution avec elle-même vaut 0."
assert psi(ref, rng.normal(0, 1, 3000)) < 0.05 and psi(ref, rng.normal(1, 1, 3000)) > 0.2, "Un échantillon de même loi doit donner un PSI faible ; un décalage d'un écart-type, un PSI fort."
assert np.isfinite(psi(ref, np.full(100, 50.0))), "Des intervalles vides ne doivent pas donner l'infini : np.clip à 1e-4."
print("Bravo, tout est correct.")

### Exercice 2 · Un rapport de dérive

La fonction `psi` est fournie, ainsi que `reference` (mois 0) et `actuel` (mois 5) du jeu de paiements. Construis `rapport`, un DataFrame indexé par les colonnes de `COLONNES`, avec : `psi`, `p_valeur` (de `scipy.stats.ks_2samp`) et `derive`, un booléen vrai si le PSI dépasse 0,1 **ou** si la p-valeur est inférieure à 0,01 **et** le PSI dépasse 0,05 (on ignore les écarts significatifs mais minuscules). Puis `colonnes_en_derive`, la liste triée des colonnes en dérive.

<details><summary>Indice</summary>

Une boucle sur COLONNES qui remplit un dictionnaire de lignes, puis pd.DataFrame(lignes).T. La règle : v > 0.1 or (p < 0.01 and v > 0.05).
</details>

In [ ]:
import numpy as np

def psi(reference, actuel, n_bins=10):
    bords = np.quantile(reference, np.linspace(0, 1, n_bins + 1))
    bords[0], bords[-1] = -np.inf, np.inf
    r = np.histogram(reference, bords)[0] / len(reference)
    a = np.histogram(actuel, bords)[0] / len(actuel)
    r, a = np.clip(r, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - r) * np.log(a / r)))
import pandas as pd
from scipy.stats import ks_2samp
from paiements import charger

reference, actuel = charger(mois=0), charger(n=4000, mois=5)
COLONNES = ["montant", "heure", "nb_tx_24h", "anciennete_compte_j"]

rapport = None
colonnes_en_derive = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from scipy.stats import ks_2samp
assert rapport is not None and sorted(rapport.index) == sorted(COLONNES), "rapport doit avoir une ligne par colonne de COLONNES."
for c in COLONNES:
    v, p = psi(reference[c], actuel[c]), ks_2samp(reference[c], actuel[c]).pvalue
    assert abs(float(rapport.loc[c, "psi"]) - v) < 1e-9 and abs(float(rapport.loc[c, "p_valeur"]) - p) < 1e-12, f"psi ou p_valeur faux pour {c}."
    assert bool(rapport.loc[c, "derive"]) == (v > 0.1 or (p < 0.01 and v > 0.05)), f"derive faux pour {c} : PSI > 0,1, ou p < 0,01 avec PSI > 0,05."
assert colonnes_en_derive == ["montant"], f"Au mois 5, seul le montant dérive vraiment ; tu as {colonnes_en_derive}."
print("Bravo, tout est correct.")

---
## Projet · MLOps de bout en bout : surveiller, réentraîner, promouvoir

Six mois ont passé depuis la mise en production du modèle anti-fraude de la vague 6. Les données de chaque mois sont arrivées, avec leurs étiquettes. Ton rôle de Lead : mettre en place la surveillance, détecter le problème, réentraîner, et décider de la promotion selon des règles écrites à l'avance.

In [ ]:
from paiements import charger
for mois in [0, 3, 6]:
    d = charger(n=4000, mois=mois)
    print(f"mois {mois} : {(d.pays_carte != d.pays_ip).mean():.0%} de paiements depuis l'étranger, {d.fraude.mean():.1%} de fraudes")

Dans les exercices, `preparer`, `psi`, la `reference` (mois 0) et le modèle `champion` (entraîné sur le mois 0) sont fournis.

**À faire :**

- [ ] Générer un rapport Evidently entre le mois 0 et le mois 5 et l'ouvrir dans le navigateur
- [ ] Écrire un DAG Airflow (ou un workflow GitHub Actions planifié) mensuel qui calcule le tableau de surveillance et envoie une alerte
- [ ] Enregistrer champion et challenger dans le registre MLflow et déplacer l'alias champion après la décision
- [ ] Rédiger une note d'une page pour un comité : ce qui s'est passé, la décision, et la règle de surveillance proposée

### Exercice 1 · Le tableau de surveillance

Construis `suivi`, un DataFrame avec une ligne par mois de 1 à 6 (les données de chaque mois : `charger(n=4000, mois=m)`) et les colonnes `mois`, `psi_montant` (PSI du montant par rapport à la référence), `part_etranger` (part des paiements où `pays_carte` ≠ `pays_ip`) et `auc` (AUC du champion sur ce mois). Puis `mois_alerte_perf`, le premier mois où l'AUC passe sous 0,9, et `mois_alerte_derive`, le premier mois où le PSI du montant dépasse 0,1.

<details><summary>Indice</summary>

Une boucle sur range(1, 7) qui remplit une liste de dictionnaires, puis pd.DataFrame. suivi.loc[condition, "mois"].min() donne le premier mois concerné.
</details>

In [ ]:
import numpy as np
import pandas as pd
from paiements import charger
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def preparer(df):
    return pd.DataFrame({
        "pays_diff": (df["pays_carte"] != df["pays_ip"]).astype(int),
        "log_montant": np.log1p(df["montant"] / 100),
        "nb_tx_24h": df["nb_tx_24h"],
        "compte_recent": (df["anciennete_compte_j"] < 30).astype(int),
        "appareil_nouveau": df["appareil_nouveau"],
        "nuit": (df["heure"] < 6).astype(int),
        "cat_risque": df["categorie"].isin(["électronique", "jeux"]).astype(int),
    }, index=df.index)
import numpy as np

def psi(reference, actuel, n_bins=10):
    bords = np.quantile(reference, np.linspace(0, 1, n_bins + 1))
    bords[0], bords[-1] = -np.inf, np.inf
    r = np.histogram(reference, bords)[0] / len(reference)
    a = np.histogram(actuel, bords)[0] / len(actuel)
    r, a = np.clip(r, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - r) * np.log(a / r)))
from sklearn.metrics import roc_auc_score

reference = charger()                                              # mois 0 : les données d'entraînement
champion = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(preparer(reference), reference["fraude"])

suivi = None
mois_alerte_perf = None
mois_alerte_derive = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
from sklearn.metrics import roc_auc_score
assert suivi is not None and list(suivi["mois"]) == [1, 2, 3, 4, 5, 6], "suivi doit avoir une ligne par mois, de 1 à 6."
for m in [2, 5]:
    d = charger(n=4000, mois=m); l = suivi[suivi["mois"] == m].iloc[0]
    assert abs(l["psi_montant"] - psi(reference["montant"], d["montant"])) < 1e-9, f"psi_montant faux au mois {m}."
    assert abs(l["part_etranger"] - (d["pays_carte"] != d["pays_ip"]).mean()) < 1e-12, f"part_etranger faux au mois {m}."
    assert abs(l["auc"] - roc_auc_score(d["fraude"], champion.predict_proba(preparer(d))[:, 1])) < 1e-9, f"auc faux au mois {m}."
assert mois_alerte_perf == 5, f"L'AUC passe sous 0,9 au mois 5 ; tu as {mois_alerte_perf}."
assert mois_alerte_derive == 6, f"Le PSI du montant dépasse 0,1 au mois 6 ; tu as {mois_alerte_derive}. La performance chute avant que la dérive des données ne sonne : c'est une dérive de concept."
print("Bravo, tout est correct.")

### Exercice 2 · Réentraîner et décider

Analyse : depuis le mois 3, des fraudes apparaissent sur la catégorie « services ». Écris `preparer_v2(df)` qui reprend `preparer(df)` et ajoute la colonne `cat_services` (1 si la catégorie est « services »). Entraîne `challenger` (même pipeline que le champion) sur les mois 4 et 5 concaténés (`charger(n=4000, mois=4)` et `mois=5`). Évalue sur le mois 6 : `auc_champion` (avec `preparer`) et `auc_challenger` (avec `preparer_v2`). Enfin `promouvoir`, True si le challenger gagne au moins 0,02 point d'AUC.

<details><summary>Indice</summary>

pd.concat([charger(n=4000, mois=4), charger(n=4000, mois=5)]) ; dans preparer_v2, X = preparer(df) puis X["cat_services"] = …
</details>

In [ ]:
import numpy as np
import pandas as pd
from paiements import charger
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def preparer(df):
    return pd.DataFrame({
        "pays_diff": (df["pays_carte"] != df["pays_ip"]).astype(int),
        "log_montant": np.log1p(df["montant"] / 100),
        "nb_tx_24h": df["nb_tx_24h"],
        "compte_recent": (df["anciennete_compte_j"] < 30).astype(int),
        "appareil_nouveau": df["appareil_nouveau"],
        "nuit": (df["heure"] < 6).astype(int),
        "cat_risque": df["categorie"].isin(["électronique", "jeux"]).astype(int),
    }, index=df.index)
import numpy as np

def psi(reference, actuel, n_bins=10):
    bords = np.quantile(reference, np.linspace(0, 1, n_bins + 1))
    bords[0], bords[-1] = -np.inf, np.inf
    r = np.histogram(reference, bords)[0] / len(reference)
    a = np.histogram(actuel, bords)[0] / len(actuel)
    r, a = np.clip(r, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - r) * np.log(a / r)))
from sklearn.metrics import roc_auc_score

reference = charger()                                              # mois 0 : les données d'entraînement
champion = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(preparer(reference), reference["fraude"])

def preparer_v2(df):
    pass

challenger = None
auc_champion = auc_challenger = None
promouvoir = None

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
d = charger(n=50, mois=6)
X2 = preparer_v2(d)
assert X2 is not None and list(X2.columns) == list(preparer(d).columns) + ["cat_services"], "preparer_v2 reprend les colonnes de preparer et ajoute cat_services à la fin."
assert (X2["cat_services"] == (d["categorie"] == "services")).all(), "cat_services vaut 1 pour la catégorie services."
test = charger(n=4000, mois=6)
assert abs(auc_champion - roc_auc_score(test["fraude"], champion.predict_proba(preparer(test))[:, 1])) < 1e-9, "auc_champion : le champion évalué sur le mois 6."
assert abs(auc_challenger - roc_auc_score(test["fraude"], challenger.predict_proba(preparer_v2(test))[:, 1])) < 1e-9, "auc_challenger : le challenger évalué sur le mois 6, avec preparer_v2."
assert auc_challenger > 0.92 and auc_champion < 0.86, f"Le challenger devrait dépasser 0,92 et le champion rester sous 0,86 ; tu as {auc_challenger:.3f} et {auc_champion:.3f}. Entraîne bien sur les mois 4 et 5."
assert promouvoir is True or promouvoir == True, "Le challenger gagne plus de 0,02 point : il doit être promu."
print("Bravo, tout est correct.")